In [2]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 598.1 kB/s  0:00:04 eta 0:00:01


In [2]:
import numpy as np
import pandas as pd
from faker import Faker
from datetime import timedelta

In [3]:
SEED= 42
NUM_PATIENTS= 5000
MAX_ADMISSIONS= 4
START_DATE= '2019-01-01'
END_DATE= '2024-12-31'

np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

In [11]:
AGE_GROUPS= {
    '18-44': 0.20,
    '45-64': 0.30,
    '65-74': 0.25,
    '75+': 0.25
}
SEX_DIST= {'Female' : 0.52, 'Male' : 0.48}

RACE_DIST= {
    'White': 0.55,
    'Black': 0.20,
    'Hispanic': 0.15,
    'Other': 0.10
}

DIAGNOSIS_DIST= {
    'Heart Failure': 0.20,
    'Pnuemonia': 0.15,
    'COPD': 0.15,
    'AMI': 0.10,
    'Stroke': 0.10,
    'Other': 0.30
}

LOS_MEANS= {
    'Heart Failure': 6,
    'Pneumonia': 5,
    'COPD':5,
    'AMI':4,
    'Stroke':7,
    'Other':4
}

In [16]:
def weighted_choice(distribution):
    return np.random.choice(
        list(distribution.keys()),
        p=list(distribution.values())
    )

def assign_payer(age_group):
    if age_group in ['65-74', '75+']:
        return np.random.choice(
            ['Medicare', 'Commercial', 'Medicaid'],
            p= [0.75, 0.15, 0.10]
        )
    else :
        return np.random.choice(
            ['Commercial', 'Medicaid','Self-pay'],
            p= [0.65, 0.25, 0.10]
        )

def generate_length_of_stay(diagnosis):
    diagnosis= str(diagnosis)
    mean_los= LOS_MEANS.get(diagnosis, LOS_MEANS['Other'])
    los= int(np.random.gamma(shape=2, scale= mean_los/2))
    return max(1, min(los, 30))

def readmission_probability(age_group, payer, diagnosis, los, prior_90d):
    prob= 0.12

    if age_group == '75+':
        prob += 0.05
    if payer == 'Medicare':
        prob += 0.03
    if diagnosis in ['Heart Failure', 'COPD']:
        prob += 0.05
    if los > 7:
        prob += 0.04
    if prior_90d:
        prob += 0.06

    return min(prob, 0.40)
    


In [17]:
records= []
patient_id= 1

start= pd.to_datetime(START_DATE)
end= pd.to_datetime(END_DATE)

for _ in range(NUM_PATIENTS):
    age_group= weighted_choice(AGE_GROUPS)
    sex= weighted_choice(SEX_DIST)
    race= weighted_choice(RACE_DIST)
    payer= assign_payer(age_group)

    num_admissions= np.random.randint(1, MAX_ADMISSIONS + 1)
    last_discharge= None

    for adm in range(num_admissions):
        if last_discharge:
            gap_days= np.random.randint(30, 365)
            admission_date= last_discharge + timedelta(days=gap_days)
        else:
            admission_date= fake.date_between(start_date= start, end_date= end)

        diagnosis= str(weighted_choice(DIAGNOSIS_DIST))
        los= generate_length_of_stay(diagnosis)
        discharge_date= admission_date + timedelta(days=los)

        prior_90d= (
            last_discharge is not None and 
            (admission_date - last_discharge).days <= 90
        )

        prob = readmission_probability(
             age_group, payer, diagnosis, los, prior_90d
        )

        readmitted_30d= int(
            np.random.rand() < prob and adm < num_admissions - 1
        )

        records.append({
             'patient_id': patient_id,
             'age_group' : age_group,
             'sex' : sex,
             'race': race,
             'payer': payer,
             'admission_date': admission_date,
             'discharge_date':discharge_date,
             'length_of_stay': los,
             'diagnosis_group':diagnosis,
             'readmitted_30d':readmitted_30d
        })

        last_discharge= discharge_date

    patient_id += 1

In [18]:
df= pd.DataFrame(records)

df.sort_values(
    ['patient_id', 'admission_date'],
    inplace= True
)

In [19]:
df.to_csv('Synthetic_Hospital_Readmissions.csv', index=False)

print('Dataset created successfully!')
print(f'Total admissions: {len(df)}')
print(f'Overall readmissions rate: {df['readmitted_30d'].mean():.2%}')

Dataset created successfully!
Total admissions: 12665
Overall readmissions rate: 10.69%
